# Troubleshooting workflows

In [0]:
file_path = '/Volumes/main/default/raw_data/2m CC Records.csv'
df_cc = spark.read.format('csv').option('header', 'true').load(file_path)

print('Dataset schema: ')
df_cc.printSchema()
df_cc.show(5)

In [0]:
from pyspark.sql.functions import count, avg, col, rand, sum
# Perform an operation that will be affected by this skew
skewed_result = df_cc.groupBy('Issuing Bank').agg(count("*").alias('card_count'), 
         avg(col('Credit Limit').cast('double')).
         alias('avg_credit_limit'))

# Force execution
skewed_result.count()

In [0]:
# Add a salt column for better distribution
df_cc_salted = df_cc.withColumn('salt', (rand() *10).cast('int'))

# USe salting technique to distribute the skewed data
balanced_result = df_cc_salted.groupBy('Issuing Bank', 'salt') \
    .agg(count("*").alias('card_count'), 
        avg(col('Credit Limit').cast('double')).alias('avg_limit')) \
    .groupBy('Issuing Bank') \
    .agg(sum('card_count').alias('total_cards'),
         avg('avg_limit').alias('avg_credit_limit'))

# Force execution
balanced_result.count()

In [0]:
# Other improvements
# Repartition
# Caching